# Site Selection Dashboard — Trade Area Isochrones

Generate drive-time isochrones around 5 candidate retail locations and export
the trade area polygons as GeoJSON, ready to render on a frontend map layer.

## 1. Import Required Libraries

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import json
import os

## 2. Configure Wherobots Session

In [ ]:
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 3. Define Candidate Retail Locations

Five candidate sites across the San Francisco Bay Area.

In [ ]:
candidates = [
    ("Union Square, SF",    -122.4075, 37.7880),
    ("Downtown Oakland",    -122.2712, 37.8044),
    ("Palo Alto",           -122.1430, 37.4419),
    ("Walnut Creek",        -122.0652, 37.9101),
    ("Downtown San Jose",   -121.8863, 37.3382),
]

schema = StructType([
    StructField("site_name", StringType()),
    StructField("lon", DoubleType()),
    StructField("lat", DoubleType()),
])

sites_df = sedona.createDataFrame(candidates, schema) \
    .withColumn("geometry", f.expr("ST_Point(lon, lat)"))

sites_df.show(truncate=False)

## 4. Generate 10-Minute Drive-Time Isochrones

Use `ST_Isochrone(point, minutes, 'car', false)` to compute the outbound
10-minute trade area polygon for each candidate site.

In [ ]:
sites_df.createOrReplaceTempView("sites")

isochrones_df = sedona.sql("""
    SELECT
        site_name,
        lon,
        lat,
        geometry AS center,
        ST_Isochrone(geometry, 10, 'car', false) AS geometry,
        ST_Area(
            ST_Transform(ST_Isochrone(geometry, 10, 'car', false), 'EPSG:4326', 'EPSG:3857')
        ) / 1e6 AS approx_area_sq_km
    FROM sites
""").cache()

isochrones_df.show(truncate=60)

## 5. Convert to GeoJSON

Add GeoJSON columns for both the center point and the trade area polygon.

In [ ]:
trade_areas_df = isochrones_df \
    .withColumn("center_geojson", f.expr("ST_AsGeoJSON(center)")) \
    .withColumn("trade_area_geojson", f.expr("ST_AsGeoJSON(geometry)"))

trade_areas_df.select("site_name", "approx_area_sq_km", "center_geojson").show(truncate=60)

## 6. Build & Export GeoJSON FeatureCollection

Assemble a standard GeoJSON `FeatureCollection` and write it to Wherobots
Managed Storage via Spark so the frontend map layer can fetch it directly.

In [ ]:
rows = trade_areas_df.select(
    "site_name", "lon", "lat", "approx_area_sq_km",
    "center_geojson", "trade_area_geojson"
).collect()

features = []
for row in rows:
    features.append({
        "type": "Feature",
        "properties": {
            "site_name":       row.site_name,
            "center_lon":      row.lon,
            "center_lat":      row.lat,
            "drive_minutes":   10,
            "approx_area_km2": round(row.approx_area_sq_km, 2),
        },
        "geometry": json.loads(row.trade_area_geojson),
    })

feature_collection = {"type": "FeatureCollection", "features": features}

# Write locally — use Spark .text() for S3 paths instead of open()
output_path = "/tmp/trade_areas.geojson"
with open(output_path, "w") as fh:
    json.dump(feature_collection, fh)

print(f"Wrote {len(features)} trade area polygons to {output_path}")
print(json.dumps(feature_collection, indent=2)[:2000], "...")

## 7. Preview on a Map

Render trade areas with `SedonaKepler` for a quick visual check before
plugging into the site selection dashboard.

In [ ]:
SedonaKepler.create_map(trade_areas_df.drop("center"), name="Trade Areas")